# 在 Responses API 中使用文件搜索工具

尽管 RAG 可能让人望而生畏，但在 PDF 文件中进行搜索不应该很复杂。目前最常用的做法之一是：解析 PDF、制定分块策略、将这些分块上传到存储服务、对这些文本块计算嵌入，并将嵌入存储到向量数据库中。而这还只是准备工作——在 LLM 工作流中检索内容同样需要多个步骤。

这正是文件搜索（file search）的用武之地——它是一个可以在 Responses API 中使用的托管工具。它能让你搜索知识库，并根据检索到的内容生成答案。在本 cookbook 中，我们会把这些 PDF 上传到 OpenAI 上的向量存储，并使用文件搜索从该向量存储中获取额外上下文，来回答我们在第一步中生成的问题。随后，我们会先基于从 OpenAI 博客（[openai.com/news](https://openai.com/news)）中提取的 PDF 创建一小组问题。

_文件搜索此前仅在 Assistants API 中提供。现在它也可以在新的 Responses API 中使用，该 API 既可以是有状态的，也可以是无状态的，并带有元数据过滤等新功能_

# 用我们的 PDF 创建向量存储

In [ ]:
!pip install PyPDF2 pandas tqdm openai -q

In [1]:
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import concurrent
import PyPDF2
import os
import pandas as pd
import base64

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
dir_pdfs = 'openai_blog_pdfs' # have those PDFs stored locally here
pdf_files = [os.path.join(dir_pdfs, f) for f in os.listdir(dir_pdfs)]

我们将在 OpenAI API 上创建一个向量存储（Vector Store），并将 PDF 上传到其中。OpenAI 会读取这些 PDF，将内容拆分成多个文本块，对它们计算嵌入，并将嵌入和文本存储到向量存储中。这样我们就可以查询该向量存储，根据查询返回相关内容。

In [2]:
def upload_single_pdf(file_path: str, vector_store_id: str):
    file_name = os.path.basename(file_path)
    try:
        file_response = client.files.create(file=open(file_path, 'rb'), purpose="assistants")
        attach_response = client.vector_stores.files.create(
            vector_store_id=vector_store_id,
            file_id=file_response.id
        )
        return {"file": file_name, "status": "success"}
    except Exception as e:
        print(f"Error with {file_name}: {str(e)}")
        return {"file": file_name, "status": "failed", "error": str(e)}

def upload_pdf_files_to_vector_store(vector_store_id: str):
    pdf_files = [os.path.join(dir_pdfs, f) for f in os.listdir(dir_pdfs)]
    stats = {"total_files": len(pdf_files), "successful_uploads": 0, "failed_uploads": 0, "errors": []}
    
    print(f"{len(pdf_files)} PDF files to process. Uploading in parallel...")

    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(upload_single_pdf, file_path, vector_store_id): file_path for file_path in pdf_files}
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(pdf_files)):
            result = future.result()
            if result["status"] == "success":
                stats["successful_uploads"] += 1
            else:
                stats["failed_uploads"] += 1
                stats["errors"].append(result)

    return stats

def create_vector_store(store_name: str) -> dict:
    try:
        vector_store = client.vector_stores.create(name=store_name)
        details = {
            "id": vector_store.id,
            "name": vector_store.name,
            "created_at": vector_store.created_at,
            "file_count": vector_store.file_counts.completed
        }
        print("Vector store created:", details)
        return details
    except Exception as e:
        print(f"Error creating vector store: {e}")
        return {}

In [3]:
store_name = "openai_blog_store"
vector_store_details = create_vector_store(store_name)
upload_pdf_files_to_vector_store(vector_store_details["id"])

Vector store created: {'id': 'vs_67d06b9b9a9c8191bafd456cf2364ce3', 'name': 'openai_blog_store', 'created_at': 1741712283, 'file_count': 0}
21 PDF files to process. Uploading in parallel...


100%|███████████████████████████████| 21/21 [00:09<00:00,  2.32it/s]


{'total_files': 21,
 'successful_uploads': 21,
 'failed_uploads': 0,
 'errors': []}

# 独立的向量搜索

向量存储准备就绪后，我们就可以直接查询它，并为特定查询检索相关内容。借助新的[向量搜索 API](https://platform.openai.com/docs/api-reference/vector-stores/search)，我们无需将其集成到 LLM 查询中，就能从知识库中找到相关条目。

In [4]:
query = "What's Deep Research?"
search_results = client.vector_stores.search(
    vector_store_id=vector_store_details['id'],
    query=query
)

In [5]:
for result in search_results.data:
    print(str(len(result.content[0].text)) + ' of character of content from ' + result.filename + ' with a relevant score of ' + str(result.score))

3502 of character of content from Introducing deep research _ OpenAI.pdf with a relevant score of 0.9813588865322393
3493 of character of content from Introducing deep research _ OpenAI.pdf with a relevant score of 0.9522476825143714
3634 of character of content from Introducing deep research _ OpenAI.pdf with a relevant score of 0.9397930296526796
2774 of character of content from Introducing deep research _ OpenAI.pdf with a relevant score of 0.9101975747303771
3474 of character of content from Deep research System Card _ OpenAI.pdf with a relevant score of 0.9036647613464299
3123 of character of content from Introducing deep research _ OpenAI.pdf with a relevant score of 0.887120981288272
3343 of character of content from Introducing deep research _ OpenAI.pdf with a relevant score of 0.8448454849432881
3262 of character of content from Introducing deep research _ OpenAI.pdf with a relevant score of 0.791345286655509
3271 of character of content from Introducing deep research _ Open

可以看到，搜索查询返回了不同大小（底层也是不同文本）的结果。它们各自的相关性分数不同，这些分数由我们使用混合搜索的排序器计算得出。

# 在单次 API 调用中将搜索结果与 LLM 结合

不过，与其先查询向量存储、再把数据传入 Responses 或 Chat Completion API 调用，在 LLM 查询中使用这些搜索结果的一种更便捷的方式，是直接把 file_search 工具作为 OpenAI Responses API 的一部分接入。

In [7]:
query = "What's Deep Research?"
response = client.responses.create(
    input= query,
    model="gpt-4o-mini",
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store_details['id']],
    }]
)

# Extract annotations from the response
annotations = response.output[1].content[0].annotations
    
# Get top-k retrieved filenames
retrieved_files = set([result.filename for result in annotations])

print(f'Files used: {retrieved_files}')
print('Response:')
print(response.output[1].content[0].text) # 0 being the filesearch call

Files used: {'Introducing deep research _ OpenAI.pdf'}
Response:
Deep Research is a new capability introduced by OpenAI that allows users to conduct complex, multi-step research tasks on the internet efficiently. Key features include:

1. **Autonomous Research**: Deep Research acts as an independent agent that synthesizes vast amounts of information across the web, enabling users to receive comprehensive reports similar to those produced by a research analyst.

2. **Multi-Step Reasoning**: It performs deep analysis by finding, interpreting, and synthesizing data from various sources, including text, images, and PDFs.

3. **Application Areas**: Especially useful for professionals in fields such as finance, science, policy, and engineering, as well as for consumers seeking detailed information for purchases.

4. **Efficiency**: The output is fully documented with citations, making it easy to verify information, and it significantly speeds up research processes that would otherwise take h

可以看到，`gpt-4o-mini` 能够回答一个需要关于 OpenAI Deep Research 的较新、较专业知识的查询。它使用了文件 `Introducing deep research _ OpenAI.pdf` 中最相关的文本块。如果想更深入地分析检索到的文本块，还可以在查询中加入 `include=["output[*].file_search_call.search_results"]`，来分析搜索引擎返回的各段文本。

# 评估性能

对于这类信息检索系统，关键在于衡量为这些答案检索到的文件的相关性和质量。本 cookbook 接下来的步骤将生成一个评估数据集，并在该数据集上计算各种指标。这种方法并不完美，我们始终建议针对你自己的用例准备经过人工验证的评估数据集，但它能向你展示评估这类系统的方法论。之所以不完美，是因为生成的某些问题可能比较笼统（例如：本文档中主要利益相关方说了什么），我们的检索测试将很难判断该问题是针对哪份文档生成的。

## 生成评估数据

我们将创建一些函数，读取本地的 PDF，并为每份文档生成一个只有该文档才能回答的问题。由此得到之后可以使用的评估数据集。

In [8]:
def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        with open(pdf_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text
    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")
    return text

def generate_questions(pdf_path):
    text = extract_text_from_pdf(pdf_path)

    prompt = (
        "Can you generate a question that can only be answered from this document?:\n"
        f"{text}\n\n"
    )

    response = client.responses.create(
        input=prompt,
        model="gpt-4o",
    )

    question = response.output[0].content[0].text

    return question

对第一个 PDF 文件运行 generate_question 函数，就可以看到它生成的问题类型。

In [9]:
generate_questions(pdf_files[0])

'What new capabilities will ChatGPT have as a result of the partnership between OpenAI and Schibsted Media Group?'

现在可以为本地存储的所有 PDF 生成问题了。

In [10]:
# Generate questions for each PDF and store in a dictionary
questions_dict = {}
for pdf_path in pdf_files:
    questions = generate_questions(pdf_path)
    questions_dict[os.path.basename(pdf_path)] = questions

In [11]:
questions_dict

{'OpenAI partners with Schibsted Media Group _ OpenAI.pdf': 'What is the purpose of the partnership between Schibsted Media Group and OpenAI announced on February 10, 2025?',
 'OpenAI and the CSU system bring AI to 500,000 students & faculty _ OpenAI.pdf': 'What significant milestone did the California State University system achieve by partnering with OpenAI, making it the first of its kind in the United States?',
 '1,000 Scientist AI Jam Session _ OpenAI.pdf': 'What was the specific AI model used during the "1,000 Scientist AI Jam Session" event across the nine national labs?',
 'Announcing The Stargate Project _ OpenAI.pdf': 'What are the initial equity funders and lead partners in The Stargate Project announced by OpenAI, and who holds the financial and operational responsibilities?',
 'Introducing Operator _ OpenAI.pdf': 'What is the name of the new model that powers the Operator agent introduced by OpenAI?',
 'Introducing NextGenAI _ OpenAI.pdf': 'What major initiative did OpenAI

现在我们有了一个 `filename:question` 字典，可以遍历它，在不提供文档的情况下向 gpt-4o(-mini) 提问，gpt-4o 应该能在向量存储中找到相关文档。

## 评估

我们将字典转换为 dataframe，并使用 gpt-4o-mini 进行处理。我们会关注预期的文件 

In [12]:
rows = []
for filename, query in questions_dict.items():
    rows.append({"query": query, "_id": filename.replace(".pdf", "")})

# Metrics evaluation parameters
k = 5
total_queries = len(rows)
correct_retrievals_at_k = 0
reciprocal_ranks = []
average_precisions = []

def process_query(row):
    query = row['query']
    expected_filename = row['_id'] + '.pdf'
    # Call file_search via Responses API
    response = client.responses.create(
        input=query,
        model="gpt-4o-mini",
        tools=[{
            "type": "file_search",
            "vector_store_ids": [vector_store_details['id']],
            "max_num_results": k,
        }],
        tool_choice="required" # it will force the file_search, while not necessary, it's better to enforce it as this is what we're testing
    )
    # Extract annotations from the response
    annotations = None
    if hasattr(response.output[1], 'content') and response.output[1].content:
        annotations = response.output[1].content[0].annotations
    elif hasattr(response.output[1], 'annotations'):
        annotations = response.output[1].annotations

    if annotations is None:
        print(f"No annotations for query: {query}")
        return False, 0, 0

    # Get top-k retrieved filenames
    retrieved_files = [result.filename for result in annotations[:k]]
    if expected_filename in retrieved_files:
        rank = retrieved_files.index(expected_filename) + 1
        rr = 1 / rank
        correct = True
    else:
        rr = 0
        correct = False

    # Calculate Average Precision
    precisions = []
    num_relevant = 0
    for i, fname in enumerate(retrieved_files):
        if fname == expected_filename:
            num_relevant += 1
            precisions.append(num_relevant / (i + 1))
    avg_precision = sum(precisions) / len(precisions) if precisions else 0
    
    if expected_filename not in retrieved_files:
        print("Expected file NOT found in the retrieved files!")
        
    if retrieved_files and retrieved_files[0] != expected_filename:
        print(f"Query: {query}")
        print(f"Expected file: {expected_filename}")
        print(f"First retrieved file: {retrieved_files[0]}")
        print(f"Retrieved files: {retrieved_files}")
        print("-" * 50)
    
    
    return correct, rr, avg_precision

In [13]:
process_query(rows[0])

(True, 1.0, 1.0)

在这个示例中，召回率和精确率都为 1，而且我们的文件排在第一位，因此该示例的 MRR 和 MAP 均为 1。

现在可以对整组问题执行这一处理。

In [14]:
with ThreadPoolExecutor() as executor:
    results = list(tqdm(executor.map(process_query, rows), total=total_queries))

correct_retrievals_at_k = 0
reciprocal_ranks = []
average_precisions = []

for correct, rr, avg_precision in results:
    if correct:
        correct_retrievals_at_k += 1
    reciprocal_ranks.append(rr)
    average_precisions.append(avg_precision)

recall_at_k = correct_retrievals_at_k / total_queries
precision_at_k = recall_at_k  # In this context, same as recall
mrr = sum(reciprocal_ranks) / total_queries
map_score = sum(average_precisions) / total_queries

 62%|███████████████████▏           | 13/21 [00:07<00:03,  2.57it/s]

Expected file NOT found in the retrieved files!
Query: What is OpenAI's mission as stated in the document?
Expected file: Disrupting malicious uses of AI _ OpenAI.pdf
First retrieved file: Introducing the Intelligence Age _ OpenAI.pdf
Retrieved files: ['Introducing the Intelligence Age _ OpenAI.pdf']
--------------------------------------------------


 71%|██████████████████████▏        | 15/21 [00:14<00:06,  1.04s/it]

Expected file NOT found in the retrieved files!
Query: What is the purpose of the "Model Spec" document published by OpenAI for ChatGPT?
Expected file: The power of personalized AI _ OpenAI.pdf
First retrieved file: Sharing the latest Model Spec _ OpenAI.pdf
Retrieved files: ['Sharing the latest Model Spec _ OpenAI.pdf', 'Sharing the latest Model Spec _ OpenAI.pdf', 'Sharing the latest Model Spec _ OpenAI.pdf', 'Sharing the latest Model Spec _ OpenAI.pdf', 'Sharing the latest Model Spec _ OpenAI.pdf']
--------------------------------------------------


100%|███████████████████████████████| 21/21 [00:15<00:00,  1.38it/s]


上面记录的输出要么表明某个文件在评估数据集期望它排第一时没有排在第一位，要么表明它根本没有被找到。从我们这个不完美的评估数据集中可以看到，有些问题比较笼统，期望的是另一份文档，而我们的检索系统并没有专门为该问题检索到它。

In [15]:
# Print the metrics with k
print(f"Metrics at k={k}:")
print(f"Recall@{k}: {recall_at_k:.4f}")
print(f"Precision@{k}: {precision_at_k:.4f}")
print(f"Mean Reciprocal Rank (MRR): {mrr:.4f}")
print(f"Mean Average Precision (MAP): {map_score:.4f}")

Metrics at k=5:
Recall@5: 0.9048
Precision@5: 0.9048
Mean Reciprocal Rank (MRR): 0.9048
Mean Average Precision (MAP): 0.8954


通过本 cookbook，我们了解了如何：
- 使用 PDF 上下文填充（利用 4o 的视觉模态）和传统 PDF 阅读器生成评估数据集
- 创建向量存储并用 PDF 填充
- 借助 OpenAI Response API 中开箱即用的 `file_search` 工具调用所提供的 RAG 系统，让 LLM 回答查询
- 理解文本块如何在 Response API 中被检索、排序和使用
- 在之前生成的评估数据集上衡量准确率、精确率、检索效果、MRR 和 MAP

通过在 Responses 中使用文件搜索，你可以简化 RAG 架构，并借助新的 Responses API 在单次 API 调用中完成这一切。文件存储、嵌入、检索全部集成在一个工具中！